# 04 — Experiment log explorer (read-only)

Browse eval records as a table: which runs exist, how their scores compare, and how a task moved over time.

| `LOG` setting | Reads |
| --- | --- |
| `"notebook"` (default) | `data/runtime/notebooks/experiment_log.jsonl`, written by notebook 03 |
| `"reports"` | `reports/experiment_log.jsonl`, the sandbox's real log (gitignored, local to this machine) |
| any other string | a JSONL path, absolute or relative to the repo root |

This notebook **only reads**. It writes nothing and needs no profile or model.

**Read with care:** compare runs only when they share a `task` **and** a `dataset_fingerprint`. Mock rows (`mock == True`) only check plumbing, so they are hidden from the charts unless `INCLUDE_MOCK = True`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Works with or without `pip install -e .`: fall back to the checkout's src/ when needed.
try:
    import mailroom_sandbox  # noqa: F401
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "src" / "mailroom_sandbox").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import mailroom_sandbox  # noqa: F401

import os

from mailroom_sandbox.notebook import bootstrap

ctx = bootstrap(create_dotenv=False, activate_profile=False)

LOG = os.environ.get("SANDBOX_NOTEBOOK_LOG_VIEW", "notebook")  # "notebook" | "reports" | a path
INCLUDE_MOCK = True  # set False to hide mock rows from the tables and charts

if LOG == "notebook":
    log_path = ctx.experiment_log
elif LOG == "reports":
    log_path = ctx.root / "reports" / "experiment_log.jsonl"
else:
    log_path = Path(LOG) if Path(LOG).is_absolute() else ctx.root / LOG
print("reading", log_path)

In [ ]:
try:
    import pandas as pd
except ImportError as exc:  # pragma: no cover - guidance for a bare [dev] install
    raise ImportError('pandas is missing: pip install -e ".[dev,notebooks]"') from exc

pd.set_option("display.max_colwidth", 80)

from mailroom_sandbox.datasets import load_jsonl

records = load_jsonl(log_path)
print(f"{len(records)} records loaded from {log_path}")
if not records:
    print("no records yet; run notebook 03 (or point LOG at another log) and rerun this cell")


## 1. Flatten records into a table

Each record carries identity fields (task, profile, provider, model, prompt version, dataset fingerprint, git snapshot) and a nested `scores` dict. The scores that sandbox tasks share are pulled into columns.

In [ ]:
SCORE_KEYS = ["exact_match", "f1_macro", "stage_correct", "extraction_overall", "routing_accuracy",
              "schema_valid_rate", "parse_error_rate"]


def flatten(record: dict) -> dict:
    scores = record.get("scores") or {}
    git = record.get("git") or {}
    row = {
        "timestamp": record.get("timestamp"),
        "experiment": record.get("experiment_name"),
        "task": record.get("task"),
        "profile": record.get("profile"),
        "provider": record.get("provider"),
        "model": record.get("model"),
        "prompt": record.get("prompt_version"),
        "mock": bool(record.get("mock")),
        "n": record.get("n"),
        "fingerprint": record.get("dataset_fingerprint"),
        "git": (git.get("sha") or git.get("commit") or "")[:10] if isinstance(git, dict) else "",
    }
    row.update({k: scores.get(k) for k in SCORE_KEYS})
    return row


runs = pd.DataFrame([flatten(r) for r in records])
if not runs.empty:
    runs["timestamp"] = pd.to_datetime(runs["timestamp"], errors="coerce", utc=True)
    runs = runs.sort_values("timestamp")
    if not INCLUDE_MOCK:
        runs = runs[~runs["mock"]]
    score_cols = [c for c in SCORE_KEYS if runs[c].notna().any()]
    print(f"{len(runs)} records, {runs['task'].nunique()} tasks, {runs['mock'].sum()} mock")
    display(runs.tail(15))

## 2. Summary by run identity

Groups by task, profile, model, mock flag and dataset fingerprint. Rows in the same group are repeat runs of the same setup.

In [ ]:
if runs.empty:
    print("nothing to summarize")
else:
    identity = ["task", "profile", "provider", "model", "prompt", "mock", "fingerprint"]
    summary = (
        runs.fillna({"model": "(default)", "profile": "?", "provider": "?", "prompt": "(default)"})
        .groupby(identity, dropna=False)
        .agg(runs=("n", "size"), docs=("n", "sum"), last=("timestamp", "max"),
             **{f"mean_{c}": (c, "mean") for c in score_cols})
        .round({f"mean_{c}": 3 for c in score_cols})
        .sort_values("last", ascending=False)
    )
    display(summary)

## 3. Score over time, per task

One line per task and dataset fingerprint for the main metric (`exact_match`). With only one run per group, there are only points.

In [ ]:
from mailroom_sandbox.notebook import show_figure
import matplotlib.pyplot as plt

metric = "exact_match"
plot_df = runs.dropna(subset=[metric]) if not runs.empty else runs
if plot_df.empty:
    print(f"no {metric} values to plot")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    for (task, fp), group in plot_df.groupby(["task", "fingerprint"], dropna=False):
        ax.plot(group["timestamp"], group[metric], marker="o", label=f"{task} · {fp}")
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} by run" + ("" if INCLUDE_MOCK else " (mock rows hidden)"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize="small", loc="lower left")
    fig.autofmt_xdate()
    fig.tight_layout()
    show_figure(fig)


## 4. Inspect one record

Set `EXPERIMENT` to an experiment name from the table above to see its latest record in full, including the per-class scores and the git snapshot.

In [ ]:
EXPERIMENT = runs["experiment"].iloc[-1] if not runs.empty else None

matches = [r for r in records if r.get("experiment_name") == EXPERIMENT]
if matches:
    print(json.dumps(matches[-1], indent=2, default=str)[:4000])
else:
    print("no matching record")